# SETUP - Community detection with GraphFrames

## Install packages and dataset - imports

In [ ]:
!pip install -q pyspark pandas networkx graphframes matplotlib
!wget https://snap.stanford.edu/data/email-Eu-core.txt.gz
!wget https://snap.stanford.edu/data/email-Eu-core-department-labels.txt.gz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 2.3 MB/s eta 0:00:00
--2026-10-02 17:29:53--  https://snap.stanford.edu/data/email-Eu-core.txt.gz
Resolving snap.stanford.edu (snap.stanford.edu)... 171.64.75.80
Connecting to snap.stanford.edu (snap.stanford.edu)|171.64.75.80|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 79754 (78K) [application/x-gzip]
Saving to: ‘email-Eu-core.txt.gz’

email-Eu-core.txt.g 100%[===================>]  77.88K   280KB/s    in 0.3s    

2026-10-02 17:29:54 (280 KB/s) - ‘email-Eu-core.txt.gz’ saved [79754/79754]

--2026-10-02 17:29:54--  https://snap.stanford.edu/data/email-Eu-core-department-labels.txt.gz
Resolving snap.stanford.edu (snap.stanford.edu)... 171.64.75.80
Connecting to snap.stanford.edu (snap.stanford.edu)|171.64.75.80|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2663 (2.6K) [application/x-gzip]
Saving to: ‘email-Eu-core-department-labels.txt.gz’

email-Eu-core-depar 100%[=====

In [ ]:
!gzip -d email-Eu-core.txt.gz
!gzip -d email-Eu-core-department-labels.txt.gz

In [ ]:
import os
os.environ["PYSPARK_SUBMIT_ARGS"] = "--packages io.graphframes:graphframes-spark4_2.13:0.12.1 pyspark-shell"

from graphframes import *
from pyspark.sql import SparkSession
from itertools import combinations
import networkx as nx
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import gzip
import time
import heapq
import glob
from collections import Counter

In [ ]:
spark = SparkSession.builder \
    .appName("CD-GF") \
    .config("spark.driver.memory", "8g") \
    .getOrCreate()

sc = spark.sparkContext

In [ ]:
print(f"Spark Version: {spark.version}")

Spark Version: 4.0.4


In [ ]:
print(f"Scala Version: {spark._jvm.scala.util.Properties.versionString()}")

Scala Version: version 2.13.16


## Dataset processing

In [ ]:
# 1. Carica e calcola pesi (puro RDD)
raw_edges = sc.textFile("email-Eu-core.txt")
data = raw_edges.map(lambda x: x.split())

# Pulisci, converti a int e aggrega
# ((u,v),w)
edges_weighted = (
    data.filter(lambda x: x[0] != x[1])
    .map(lambda x: ((min(int(x[0]), int(x[1])), max(int(x[0]), int(x[1]))), 1))
    .reduceByKey(lambda a, b: a + b)
)

# 2. Genera mapping deterministico 0...N-1 (puro RDD)
unique_nodes = (
    edges_weighted.flatMap(lambda x: [x[0][0], x[0][1]])
    .distinct()
    .sortBy(lambda x: x)
)

# RDD di tuple: (u, new_u)
node_map = unique_nodes.zipWithIndex().cache()

# 3. Rimappa gli archi con due join (stile PageRank del lab)
# (u, (v, weight))
edges_by_u = edges_weighted.map(lambda x: (x[0][0], (x[0][1], x[1])))

# Join su u -> (u, ((v, weight), new_u))
#  map -> (v, (new_u, weight))
edges_mapped_u = edges_by_u.join(node_map).map(
    lambda x: (x[1][0][0], (x[1][1], x[1][0][1]))
)

# Join su v -> (v, ((new_u, weight),new_v))
# map -> (new_u, new_v, weight)
edges_mapped = edges_mapped_u.join(node_map).map(
    lambda x: (x[1][0][0], x[1][1], x[1][0][1])
).cache()

# 4. Creazione dei DataFrame per GraphFrames, friend is needed for compatibility with paper code
# Prepariamo gli archi bidirezionali come tuple native: (src, dst, "friend", weight)
edges_rdd = edges_mapped.flatMap(
    lambda x: [(x[0], x[1], "friend", x[2]), (x[1], x[0], "friend", x[2])]
)
edges_df = edges_rdd.toDF(["src", "dst", "relationship", "weight"])

# Prepariamo i vertici come tuple native: (str(id), id, [id])
vertices_rdd = node_map.map(lambda x: (str(x[1]), x[1], [x[1]]))
vertices_df = vertices_rdd.toDF(["name", "id", "community"])

g = GraphFrame(vertices_df, edges_df)

/usr/local/lib/python3.13/dist-packages/pyspark/sql/classic/dataframe.py:146: UserWarning: DataFrame.sql_ctx is an internal property, and will be removed in future releases. Use DataFrame.sparkSession instead.
  warnings.warn(


In [ ]:
# 1. Carica Ground Truth: (id_originale, dept)
gt_raw = sc.textFile("email-Eu-core-department-labels.txt").map(
    lambda line: (int(line.split()[0]), int(line.split()[1]))
)

# 2. Join con node_map: (id_originale, (dept, id_rimappato)) -> (id_rimappato, dept)
gt_mapped = (
    gt_raw.join(node_map)
    .map(lambda x: (x[1][1], x[1][0]))
    .sortByKey()  # Ordina da 0 a N-1
)

# 3. Salva la GT rimappata in locale/storage
gt_mapped.map(lambda x: f"{x[0]},{x[1]}").saveAsTextFile("gt_mapped")

In [ ]:
# --- 1. Graph Structural Metrics ---
num_nodes = unique_nodes.count()
num_undirected_edges = edges_weighted.count()
total_weight = edges_weighted.map(lambda x: x[1]).sum()

reciprocal_edges = edges_weighted.filter(lambda x: x[1] == 2).count()
unidirectional_edges = edges_weighted.filter(lambda x: x[1] == 1).count()
graph_density = (2.0 * num_undirected_edges) / (num_nodes * (num_nodes - 1))

# Degrees
degrees = (
    edges_weighted.flatMap(lambda x: [(x[0][0], 1), (x[0][1], 1)])
    .reduceByKey(lambda a, b: a + b)
    .cache()
)

avg_degree = degrees.map(lambda x: x[1]).mean()
min_degree = degrees.map(lambda x: x[1]).min()
max_degree = degrees.map(lambda x: x[1]).max()
top_5_hubs = degrees.takeOrdered(5, key=lambda x: -x[1])

# --- 2. Ground Truth Department Metrics ---
# gt_mapped: RDD of (node_id, dept_id)
dept_counts = (
    gt_mapped.map(lambda x: (x[1], 1)).reduceByKey(lambda a, b: a + b).cache()
)

num_departments = dept_counts.count()
avg_dept_size = dept_counts.map(lambda x: x[1]).mean()
min_dept_size = dept_counts.map(lambda x: x[1]).min()
max_dept_size = dept_counts.map(lambda x: x[1]).max()

# --- Output  ---
print("==================================================")
print("             GRAPH SUMMARY (UNDIRECTED)           ")
print("==================================================")
print(f"Nodes (active / largest WCC):        {num_nodes}")
print(f"Undirected unique edges:             {num_undirected_edges}")
print(f"Total volume (sum of weights):       {total_weight}")
print(
    f"Node degree (avg [min, max]):        {avg_degree:.2f} [{min_degree},"
    f" {max_degree}]"
)
print("--------------------------------------------------")
print("             GROUND TRUTH SUMMARY                 ")
print("--------------------------------------------------")
print(f"Number of departments:               {num_departments}")
print(
    f"Community size (avg [min, max]):     {avg_dept_size:.1f} [{min_dept_size},"
    f" {max_dept_size}]"
)
print("==================================================")

             GRAPH SUMMARY (UNDIRECTED)           
Nodes (active / largest WCC):        986
Undirected unique edges:             16064
Total volume (sum of weights):       24929
Node degree (avg [min, max]):        32.58 [1, 345]
--------------------------------------------------
             GROUND TRUTH SUMMARY                 
--------------------------------------------------
Number of departments:               42
Community size (avg [min, max]):     23.5 [1, 107]


## Functions

In [ ]:
# coding: utf-8

__author__      = "Elena-Simona Apostol, Adrian-Cosmin Cojocaru, and Ciprian-Octavian Truică"
__affiliation__ = "National University of Science and Technology Politehnica Bucharest"
__copyright__   = "Copyright 2024"
__license__     = "GNU GPL"
__version__     = "0.1"
__email__       = "ciprian.truica@upb.ro"
__status__      = "Research"


# k-clicques

def get_cliques_from_motifs(motifs):
    cliques = set()
    for row in motifs:
        current_clique = []
        for x in row:
            if "src" in x:
                current_clique.append(x["src"])
        cliques.add(frozenset(current_clique))

    return list(cliques)

def get_pattern_motif(k):
    ch = 97
    nodes = []
    for i in range(k):
        x = chr(ch+i)
        nodes.append(x)

    pattern = ""
    for (src, dst) in combinations(nodes, 2):
        pattern += r'({})-[{}{}]->({});'.format(src, src, dst, dst)
        pattern += r'({})-[{}{}]->({});'.format(dst, dst, src, src)

    return pattern[:-1]


def get_clique_neighbours(clique, cliques_of):
    neighbour_cliques = set()
    for n in clique:
        for adj_clique in cliques_of[n]:
            if clique != adj_clique:
                neighbour_cliques.add(adj_clique)
    return neighbour_cliques

def run_kcliques(k, g):
    k = k
    pattern = get_pattern_motif(k)

    motifs = g.find(pattern).collect()
    cliques = get_cliques_from_motifs(motifs)
    cliques_of = {}
    for clique in cliques:
        for vertex in clique:
            if vertex not in cliques_of:
                cliques_of[vertex] = []
            cliques_of[vertex].append(clique)

    clique_nodes, clique_edges = [], []
    cliques_graph = nx.Graph()
    cliques_graph.add_nodes_from(cliques)
    for clique in cliques:
        neighbors = get_clique_neighbours(clique, cliques_of)
        for neighbor_clique in neighbors:
            common_nodes = set(clique).intersection(set(neighbor_clique))
            if len(common_nodes) == (k - 1):
                cliques_graph.add_edge(clique, neighbor_clique)


    communities = []
    for component in nx.connected_components(cliques_graph):
       communities.append(list(frozenset.union(*component)))
    for community in communities:
        print(sorted(community))
    return communities

# Fast Greedy & Louvain utils

def get_graph_vertices_list(g):
    vertices = []
    dataframe_vertices = g.vertices.select("id").collect()
    for row in dataframe_vertices:
        vertices.append(row.id)
    return vertices

def get_graph_edges_list(g):
    edges = []
    dataframe_edges = g.edges.select("src","dst","weight").collect()
    for row in dataframe_edges:
        edges.append((row.src, row.dst,row.weight))
    return edges

def get_graph_weights_sum(g):
    dataframe_weights_sum = g.edges.select('weight').groupBy().sum().collect()
    return dataframe_weights_sum[0]["sum(weight)"]

def get_nodes_degrees(g):
    nodes = get_graph_vertices_list(g)
    in_degree = {}
    out_degree = {}
    idx = 0
    for node in nodes:
        out_degree[node] = g.edges.select("weight").where("src=={}".format(node)).groupBy().sum().collect()[0]["sum(weight)"]
        in_degree[node] = g.edges.select("weight").where("dst=={}".format(node)).groupBy().sum().collect()[0]["sum(weight)"]
        if in_degree[node] == None:
            in_degree[node] = 0
        if out_degree[node] == None:
            out_degree[node] = 0
        idx += 1
    return in_degree, out_degree

# Fast Greedy

def store_delta_ij(edge, in_degree, out_degree, m, delta_Q):
    i, j, weight_ij = edge
    ki_in, ki_out, kj_in, kj_out = in_degree[i], out_degree[i], in_degree[j], out_degree[j]
    # delta_Q[(i,j)] = (weight_ij/m) - (in_degree[i] * out_degree[j] + in_degree[j] * out_degree[i])/(2*m)**2
    delta_Q_ij = (weight_ij/m) - ((in_degree[i] * out_degree[j] + in_degree[j] * out_degree[i])/(m**2))
    if not i in delta_Q:
        delta_Q[i] = {j: delta_Q_ij}
    else:
        delta_Q[i][j] = delta_Q_ij

    if not j in delta_Q:
        delta_Q[j] = {i: delta_Q_ij}
    else:
        delta_Q[j][i] = delta_Q_ij

def make_heap(delta_Q, delta_Q_heap):
    for i in delta_Q:
        delta_Q_heap[i]  = []
        for j in delta_Q[i]:
            delta_ij = delta_Q[i][j]
            delta_Q_heap[i].append((-delta_ij, i, j))
        heapq.heapify(delta_Q_heap[i])

def update_heaps(j, k, delta, delta_Q_heap, delta_Q, H, neighbor_flag=False):
    if len(delta_Q_heap[j]) > 0:
        current_peek = delta_Q_heap[j][0]
    else:
        current_peek = None
    delta_Q[j][k] = delta
    if neighbor_flag:
        delta_Q_heap[j] = list(filter(lambda x: not(x[1] == j and x[2] == k), delta_Q_heap[j]))
        heapq.heapify(delta_Q_heap[j])
        heapq.heappush(delta_Q_heap[j], (-delta, j, k))
    else:
        heapq.heappush(delta_Q_heap[j], (-delta, j, k))

    if current_peek is None:
        heapq.heappush(H, delta_Q_heap[j][0])
    elif current_peek != delta_Q_heap[j][0]:
        H.remove(current_peek)
        heapq.heapify(H)
        heapq.heappush(H, delta_Q_heap[j][0])

def remove_from_heaps(i, j, H, delta_Q, delta_Q_heap):
    if len(delta_Q_heap[i]) > 0 and delta_Q_heap[i][0][1] == i and delta_Q_heap[i][0][2] == j:
        i_row_peek = heapq.heappop(delta_Q_heap[i])
        H.remove(i_row_peek)
        heapq.heapify(H)
        if len(delta_Q_heap[i]) >= 1:
            heapq.heappush(H, delta_Q_heap[i][0])
    else:
        delta_Q_heap[i] = list(filter(lambda x: not(x[1] == i and x[2] == j), delta_Q_heap[i]))
        heapq.heapify(delta_Q_heap[i])



def remove_node(i, community, in_degree, out_degree, H, delta_Q, delta_Q_heap):

    # merge node i into community, community aquires its in/out degrees
    in_degree[community] += in_degree[i]
    out_degree[community] += out_degree[i]
    in_degree[i], out_degree[i] = 0, 0

    for j in list(delta_Q[i]):
        delta_Q[j].pop(i)
        if j == community:
            continue
        remove_from_heaps(i, j, H, delta_Q, delta_Q_heap)
        remove_from_heaps(j, i, H, delta_Q, delta_Q_heap)

    delta_Q.pop(i)

def run_graphframes(g):
    m = get_graph_weights_sum(g)
    in_degree, out_degree = get_nodes_degrees(g)
    # print(in_degree, out_degree)
    edges = get_graph_edges_list(g)
    delta_Q, delta_Q_heap = {}, {}
    for edge in edges:
        store_delta_ij(edge, in_degree, out_degree, m, delta_Q)
    make_heap(delta_Q, delta_Q_heap)
    H = []
    for (i, delta) in delta_Q_heap.items():
        H.append(delta[0])
    heapq.heapify(H)

    communities = {i: set([i]) for i in out_degree}
    yield communities.values()
    while True:
        if len(H) < 1:
            break
        delta_ij, i, j = heapq.heappop(H)
        delta_ij *= -1
        yield delta_ij
        heapq.heappop(delta_Q_heap[i])

        # push next biggest priority pair from i row heap into H
        if len(delta_Q_heap[i]) >= 1:
            heapq.heappush(H, delta_Q_heap[i][0])

        # remove (j,i) pair from H(if neccessary) and from j row heap
        if len(delta_Q_heap[j]) >= 1:
            delta_ji, j1, i1 = delta_Q_heap[j][0]
            if i == i1 and j == j1:
                H = list(filter(lambda x: not(x[1] == j1 and x[2] == i1), H))
                heapq.heapify(H)
                heapq.heappop(delta_Q_heap[j])
                #  push next biggest priority pair from j row heap into H
                if len(delta_Q_heap[j]) >= 1:
                    heapq.heappush(H, delta_Q_heap[j][0])
            else:
                delta_Q_heap[j] = list(filter(lambda x: x[1] != j and x[2] != i, delta_Q_heap[j]))
                heapq.heapify(delta_Q_heap[j])

        i_adjacent_nodes = set(delta_Q[i].keys())
        j_adjacent_nodes = set(delta_Q[j].keys())
        disjoint_neigbors = i_adjacent_nodes.union(j_adjacent_nodes)

        communities[j].update(communities[i])
        communities.pop(i)

        for k in disjoint_neigbors:
            if k == i or k == j:
                continue
            elif k in i_adjacent_nodes and k in j_adjacent_nodes:
                delta_jk = delta_Q[i][k] + delta_Q[j][k]
            elif k in i_adjacent_nodes:
                delta_jk = delta_Q[i][k] - (in_degree[j] * out_degree[k] + in_degree[k] * out_degree[j])/(m**2)
            elif k in j_adjacent_nodes:
                delta_jk = delta_Q[j][k] - (in_degree[i] * out_degree[k] + in_degree[k] * out_degree[i])/(m**2)
            update_heaps(j, k, delta_jk, delta_Q_heap, delta_Q, H, k in j_adjacent_nodes)
            update_heaps(k, j, delta_jk, delta_Q_heap, delta_Q, H, k in j_adjacent_nodes)
        remove_node(i, j, in_degree, out_degree, H, delta_Q, delta_Q_heap)

        yield communities.values()

def run_fastgreedy(g):
    community_gen = run_graphframes(g)

    cutoff = 1
    communities = next(community_gen)

    while len(communities) > 1:
        dq = next(community_gen)
        if dq < 0:
            break
        communities = next(community_gen)

    for community in communities:
        print(sorted(community))

    return communities


# Louvain

def get_degree_to_adjacent_communities(g, node, community_of):
    degree = {}
    neighbors = list(map(lambda x: (x.dst,x.weight), g.edges.select("dst", "weight").where("src=={}".format(node)).collect()))
    for (neighbor, weight) in neighbors:
        adjacent_community = community_of[neighbor]
        if adjacent_community not in degree:
            degree[adjacent_community] = 0
        degree[adjacent_community] += weight
    return degree

def get_global_community(g, node):
    enclosed_nodes = set(list(g.vertices.select("community").where("id=={}".format(node)).collect()[0].community))
    return enclosed_nodes

def calc_modularity(g, partition, m):
    m = get_graph_weights_sum(g)
    modularity = 0
    in_degree, out_degree = get_nodes_degrees(g)
    edges = get_graph_edges_list(g)
    community_of = {}
    for idx, part in enumerate(partition):
        for node in part:
            community_of[node] = idx
    for edge in edges:
        src, dst = edge[0], edge[1]
        community_src, community_dst = community_of[src], community_of[dst]
        if community_src != community_dst:
            continue
        weight = edge[2]
        modularity += (weight/2*m) - (in_degree[src]*out_degree[dst])/(2*m**2)
    return modularity

def first_phase_louvain(g, global_partition, m, gamma):
    community_of = {node: idx for idx, node in enumerate(get_graph_vertices_list(g))}
    nodes = get_graph_vertices_list(g)
    new_partition = [set() for node in nodes]
    (in_degree, out_degree), (community_in, community_out) = get_nodes_degrees(g), get_nodes_degrees(g)
    while True:
        stop = True
        for node in nodes:
            chosen_comunity = community_of[node]
            max_improvement = 0
            community_in[chosen_comunity] -= in_degree[node]
            community_out[chosen_comunity] -= out_degree[node]
            degree_to_adj_communities = get_degree_to_adjacent_communities(g, node, community_of)
            for (adjacent_community, adjacent_degree) in degree_to_adj_communities.items():
                improvement = (adjacent_degree - gamma * (in_degree[node] * community_out[adjacent_community] + out_degree[node] * community_in[adjacent_community])/m)
                if improvement > max_improvement:
                    max_improvement = improvement
                    chosen_comunity = adjacent_community
            community_in[chosen_comunity] += in_degree[node]
            community_out[chosen_comunity] += out_degree[node]
            if chosen_comunity != community_of[node]:
                community_of[node] = chosen_comunity
                stop = False
        if stop:
            break
    for node, community in community_of.items():
        new_partition[community].add(node)
        global_community = get_global_community(g, node)
        global_partition[node].difference_update(global_community)
        global_partition[community].update(global_community)
    new_partition = list(filter(lambda x: x != set(), new_partition))
    global_partition = list(filter(lambda x: x != set(), global_partition))

    return global_partition, new_partition, stop

def second_phase_louvain(g, new_partition):
    community_of = {}
    vertices_columns = ["name", "id", "community"]
    new_vertices = []
    for idx, partition in enumerate(new_partition):
        enclosed_nodes = []
        for node in partition:
            community_of[node] = idx
            sub_nodes = g.vertices.select("community").where("id=={}".format(node)).collect()[0].community
            enclosed_nodes += sub_nodes
        new_vertices.append((str(idx), idx, enclosed_nodes))

    edges = get_graph_edges_list(g)
    weights_between_communities = {}
    for edge in edges:
        src, dst, weight = edge[0], edge[1], edge[2]
        community_src, community_dst = community_of[src], community_of[dst]
        if not (community_src, community_dst) in weights_between_communities:
            weights_between_communities[(community_src, community_dst)] = 0
        weights_between_communities[(community_src, community_dst)] += weight

    new_edges = []
    edges_columns = ["src", "dst", "relationship", "weight"]
    for k, v in weights_between_communities.items():
        new_edges.append((k[0], k[1], "friend", v))

    v = spark.createDataFrame(new_vertices, vertices_columns)
    e = spark.createDataFrame(new_edges, edges_columns)
    new_g = GraphFrame(v, e)
    return new_g

def run_louvain(g, gamma=0.05):
    m = get_graph_weights_sum(g)
    communities = [{node} for node in get_graph_vertices_list(g)]
    current_modularity = calc_modularity(g, communities, m)
    threshold=0.0000001
    iteration = 0
    while True:
        communities, next_partition, stop = first_phase_louvain(g, communities, m, gamma)
        if stop:
            break

        new_mod = calc_modularity(g, next_partition, m)
        if new_mod - current_modularity <= threshold:
            break

        current_modularity = new_mod
        g = second_phase_louvain(g, next_partition)
        iteration += 1
    print(iteration)
    for community in communities:
        print(sorted(community))
    return communities

In [ ]:
def save_driver_communities(communities, output_path):
  # Appiattisce: genera coppie (nodo, id_comunita)
  flat_comms = [
      (node, comm_id)
      for comm_id, comm in enumerate(communities)
      for node in comm
  ]

  # Crea RDD, ordina e salva nello stesso formato di gt_mapped
  sc.parallelize(flat_comms).sortByKey().map(
      lambda x: f"{x[0]},{x[1]}"
  ).saveAsTextFile(output_path)

def save_df_communities(df_result, output_path):
  df_result.select("id", "label").rdd.map(
      lambda row: (row["id"], row["label"])
  ).sortByKey().map(lambda x: f"{x[0]},{x[1]}").saveAsTextFile(output_path)

def log_benchmark(algo_name, elapsed_time):
  with open("benchmark_times.txt", "a") as f:
    f.write(f"{algo_name},{elapsed_time:.4f}\n")

# Comminity Detection

## K-Cliques

In [ ]:
t1 = time.time()
comms_kcliques = run_kcliques(3, g)
t_kc = time.time() - t1
print("K-Cliques Execution Time:", t_kc)

save_driver_communities(comms_kcliques, "results_kcliques")
log_benchmark("K-Cliques", t_kc)

/usr/local/lib/python3.13/dist-packages/pyspark/sql/classic/dataframe.py:128: UserWarning: DataFrame constructor is internal. Do not directly use it.
  warnings.warn("DataFrame constructor is internal. Do not directly use it.")


[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 214, 215, 216, 217, 218, 219, 220, 221,

## Fast Greedy

In [ ]:
t1 = time.time()
comms_fg = run_fastgreedy(g)
t_fg = time.time() - t1
print("FastGreedy Execution Time:", t_fg)

save_driver_communities(comms_fg, "results_fastgreedy")
log_benchmark("FastGreedy", t_fg)

[62, 590, 722]
[308, 424, 462, 463, 561, 691, 766]
[434, 471, 472, 544, 669, 773]
[107, 468, 692, 693, 753, 815, 818]
[415, 687, 833]
[82, 600, 726, 751, 834]
[117, 160, 539, 712, 769, 802, 838, 863]
[15, 97, 98, 99, 100, 101, 125, 139, 140, 164, 185, 216, 272, 273, 274, 322, 323, 334, 335, 353, 354, 428, 429, 446, 447, 448, 461, 541, 579, 591, 616, 617, 656, 668, 678, 697, 698, 721, 744, 752, 754, 777, 781, 782, 800, 910, 914, 916, 918]
[155, 435, 512, 533, 612, 613, 615, 645, 776, 858, 901, 923]
[166, 230, 282, 366, 373, 559, 595, 728, 879, 931]
[13, 51, 78, 79, 109, 118, 128, 133, 279, 301, 402, 420, 445, 497, 557, 580, 596, 623, 639, 648, 663, 735, 783, 791, 801, 825, 829, 921, 927, 951]
[16, 45, 46, 81, 106, 114, 124, 142, 169, 173, 187, 210, 249, 254, 255, 256, 283, 300, 303, 329, 355, 356, 363, 365, 367, 371, 405, 419, 456, 460, 489, 490, 495, 518, 531, 548, 564, 577, 578, 588, 605, 606, 629, 636, 638, 640, 664, 666, 677, 732, 741, 755, 756, 779, 784, 809, 819, 857, 870, 872, 87

## Louvain

In [ ]:
t1 = time.time()
comms_louvain = run_louvain(g)
t_lv = time.time() - t1
print("Louvain Execution Time:",t_lv)

save_driver_communities(comms_louvain, "results_louvain_gamma_base")
log_benchmark("Louvain", t_lv)

0
[7, 8, 9, 11, 12, 19, 43, 44, 141, 161, 213, 246, 247, 264, 265, 266, 267, 293, 324, 332, 358, 359, 360, 374, 406, 407, 421, 441, 451, 452, 466, 487, 488, 496, 498, 499, 500, 501, 502, 503, 504, 505, 506, 510, 525, 529, 530, 555, 558, 565, 566, 569, 570, 573, 601, 607, 646, 655, 660, 665, 667, 689, 690, 696, 708, 717, 738, 749, 786, 787, 804, 808, 811, 837, 874, 893, 894, 903, 932, 937, 938, 948, 953, 977]
[0, 1, 2, 3, 4, 5, 6, 10, 13, 14, 15, 16, 17, 18, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 142, 143, 144, 145, 1

## Louvain

In [18]:
t1 = time.time()
comms_louvain = run_louvain(g, gamma = 1 )
t_lv = time.time() - t1
print("Louvain Execution Time:",t_lv)

save_driver_communities(comms_louvain, "results_louvain_gamma_1")
log_benchmark("Louvain", t_lv)

ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/usr/local/lib/python3.13/dist-packages/py4j/clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ~~~~~~~~~~~~~~~~~~~~^^
  File "/usr/lib/python3.13/socket.py", line 723, in readinto
    return self._sock.recv_into(b)
           ~~~~~~~~~~~~~~~~~~~~^^^
KeyboardInterrupt
ERROR:py4j.clientserver:Exception occurred while shutting down connection
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/usr/local/lib/python3.13/dist-packages/py4j/clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
         

KeyboardInterrupt: 

## LPA


In [20]:
t1 = time.time()
result_lpa = g.labelPropagation(maxIter=15).cache()
result_lpa.count()  # trigger the lazy evaluation
t_lpa_gf = time.time() - t1

print("LPA Execution Time:",t_lpa_gf)

save_df_communities(result_lpa, "results_lpa_gf")
log_benchmark("LPA_GraphFrames", t_lpa_gf)

LPA Execution Time: 10.90836763381958


# Results

In [21]:
import glob
import pandas as pd
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score


def load_spark_folder(folder_path, value_col_name):
  """Legge tutti i file part-* escludendo file di controllo .crc"""
  files = [
      f
      for f in glob.glob(f"{folder_path}/part-*")
      if not f.endswith(".crc") and not f.endswith("_SUCCESS")
  ]
  df = pd.concat(
      (
          pd.read_csv(f, header=None, names=["node", value_col_name])
          for f in files
      ),
      ignore_index=True,
  )
  return df.drop_duplicates(subset=["node"]).sort_values("node")


# 1. Caricamento dei dataset
gt_df = load_spark_folder("gt_mapped", "ground_truth")
fg_df = load_spark_folder("results_fastgreedy", "fastgreedy")
lpa_df = load_spark_folder("results_lpa_gf", "lpa")

# 2. Join interno per garantire allineamento riga per riga
df = gt_df.merge(fg_df, on="node").merge(lpa_df, on="node")

# 3. Calcolo metriche di confronto
summary_data = {
    "Algorithm": ["Ground Truth", "FastGreedy", "LPA (GraphFrames)"],
    "Nodes Evaluated": [len(df), len(df), len(df)],
    "Communities Detected": [
        df["ground_truth"].nunique(),
        df["fastgreedy"].nunique(),
        df["lpa"].nunique(),
    ],
    "NMI (Mutual Info)": [
        1.0,
        normalized_mutual_info_score(df["ground_truth"], df["fastgreedy"]),
        normalized_mutual_info_score(df["ground_truth"], df["lpa"]),
    ],
    "ARI (Rand Index)": [
        1.0,
        adjusted_rand_score(df["ground_truth"], df["fastgreedy"]),
        adjusted_rand_score(df["ground_truth"], df["lpa"]),
    ],
}

summary_table = pd.DataFrame(summary_data)
print(summary_table.to_string(index=False))

        Algorithm  Nodes Evaluated  Communities Detected  NMI (Mutual Info)  ARI (Rand Index)
     Ground Truth              986                    42           1.000000          1.000000
       FastGreedy              986                    21           0.579067          0.343601
LPA (GraphFrames)              986                     1           0.000000          0.000000
